In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from ultralytics import YOLO
from transformers import DetrImageProcessor, DetrForObjectDetection
from PIL import Image

# 1. Custom Dataset for .npy patches
class NumpyPatchDataset(Dataset):
    def __init__(self, file_path):
        # Load the .npy file (assuming shape [N, H, W, C])
        self.data = np.load(file_path)
        if self.data.dtype == np.float32 or self.data.dtype == np.float64:
            # Ensure data is in 0-255 range if stored as 0-1
            if self.data.max() <= 1.0:
                self.data = (self.data * 255).astype(np.uint8)
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_np = self.data[idx]
        # Convert to PIL for consistent preprocessing across models
        return Image.fromarray(img_np)

# 2. Model Loading Functions
def load_models():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # FasterRCNN
    frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    frcnn.to(device).eval()
    
    # YOLOv8
    yolo = YOLO("yolov8n.pt") # or your custom weights
    
    # DETR
    detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
    detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device).eval()
    
    return frcnn, yolo, detr_model, detr_processor, device

In [80]:
import torch
import numpy as np
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.transforms import functional as F

def get_model_preds(model_type, model_obj, img_pil, device, processor=None, threshold=0.5):
    """
    Standardizes output from different models into torchmetrics format.
    Returns: dict {'boxes': tensor, 'scores': tensor, 'labels': tensor}
    """
    if model_type == "frcnn":
        img_tensor = F.to_tensor(img_pil).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model_obj(img_tensor)[0]
        mask = out['scores'] > threshold
        return {k: v[mask].detach().cpu() for k, v in out.items()}

    elif model_type == "yolo":
        res = model_obj(img_pil, verbose=False)[0]
        # YOLOv8 boxes: xyxy, conf, cls
        mask = res.boxes.conf > threshold
        return {
            "boxes": res.boxes.xyxy[mask].detach().cpu(),
            "scores": res.boxes.conf[mask].detach().cpu(),
            "labels": res.boxes.cls[mask].detach().cpu().int()
        }

    elif model_type == "detr":
        inputs = processor(images=img_pil, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model_obj(**inputs)
        target_sizes = torch.tensor([img_pil.size[::-1]])
        out = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=threshold)[0]
        return {k: v.detach().cpu() for k, v in out.items()}

import torch
import numpy as np

def calculate_iou(boxA, boxB):
    # box format: [x1, y1, x2, y2]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

def evaluate_confidence_robustness(clean_path, test_path, label, t):
    clean_ds = NumpyPatchDataset(clean_path)
    test_ds = NumpyPatchDataset(test_path)
    frcnn, yolo, detr, detr_proc, device = load_models()
    
    # Store aggregate scores for each model
    total_metrics = {"frcnn": [], "yolo": [], "detr": []}

    print(f"\n🚀 Analyzing Confidence Robustness for {label}...")

    for idx in range(len(clean_ds)):
        img_clean = clean_ds[idx]
        img_test = test_ds[idx]

        for m_name in ["frcnn", "yolo", "detr"]:
            m_obj = {"frcnn": frcnn, "yolo": yolo, "detr": detr}[m_name]
            proc = detr_proc if m_name == "detr" else None
            
            # 1. Get baseline and test predictions
            clean_p = get_model_preds(m_name, m_obj, img_clean, device, proc, threshold=t)
            test_p = get_model_preds(m_name, m_obj, img_test, device, proc, threshold=t) # Lower for test

            conf_score = 0.0
            matched_test_indices = set()

            # 2. Compare every Clean box to Test boxes
            for i, c_box in enumerate(clean_p["boxes"]):
                best_iou = 0
                match_idx = -1
                
                for j, t_box in enumerate(test_p["boxes"]):
                    if j in matched_test_indices :
                        continue
                    if clean_p['labels'][i] == test_p['labels'][j]:
                        iou = calculate_iou(c_box, t_box)
                        if iou > 0.5 and iou > best_iou:
                            best_iou = iou
                            match_idx = j
                
                if match_idx != -1:
                    # Match found: Calculate difference (Recovery/Drop)
                    # A positive value means the model is as confident or better
                    conf_score +=  best_iou * (test_p["scores"][match_idx] / clean_p["scores"][i])
                    matched_test_indices.add(match_idx)
                else:
                    # Match NOT found: Total penalty for this object
                    conf_score += 0.0

            # 3. Handle 'Ghost' detections (Test boxes that have no Clean match)
            # These are usually False Positives caused by the patch
            num_ghosts = len(test_p["boxes"]) - len(matched_test_indices)
            
            final_img_score = (conf_score / max(len(clean_p["boxes"]), 1)) 
            total_metrics[m_name].append([max(0, final_img_score), num_ghosts])

    for items in total_metrics.items():
        print(items[0])
        avg_score = 0
        avg_ghosts = 0
        for score, ghosts in items[1]:
            avg_ghosts += ghosts
            avg_score += score
        avg_score /= len(items[1])
        avg_ghosts /= len(items[1])
        print(f'avg_score:', avg_score)
        print(f'avg_ghosts:', avg_ghosts)

    return total_metrics

In [81]:
total_metrics_patched = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_patched.npy", "Patched Set", t=0.7)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 3056.54it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Patched Set...
frcnn
avg_score: tensor(0.6475)
avg_ghosts: 1.1
yolo
avg_score: tensor(0.7073)
avg_ghosts: 0.0
detr
avg_score: tensor(0.6305)
avg_ghosts: 3.9


In [83]:
total_metrics_defended = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_defended.npy", "Defended Set", t=0.7)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 3794.53it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Defended Set...
frcnn
avg_score: tensor(0.7324)
avg_ghosts: 0.5
yolo
avg_score: tensor(0.9028)
avg_ghosts: 0.0
detr
avg_score: tensor(0.5916)
avg_ghosts: 2.4


In [86]:
total_metrics_defended = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_defended_multiplied.npy", "Defended Set", t =0.7)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 4584.52it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Defended Set...
frcnn
avg_score: tensor(0.7249)
avg_ghosts: 0.8
yolo
avg_score: tensor(0.7883)
avg_ghosts: 0.0
detr
avg_score: tensor(0.6372)
avg_ghosts: 2.3


In [88]:
total_metrics_patched = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_patched.npy", "Patched Set", t=0.5)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 4029.36it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Patched Set...
frcnn
avg_score: tensor(0.6687)
avg_ghosts: 3.0
yolo
avg_score: tensor(0.6203)
avg_ghosts: 0.3
detr
avg_score: tensor(0.6329)
avg_ghosts: 5.0


In [89]:
total_metrics_defended = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_defended.npy", "Defended Set", t=0.5)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 3569.48it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Defended Set...
frcnn
avg_score: tensor(0.6386)
avg_ghosts: 1.4
yolo
avg_score: tensor(0.7093)
avg_ghosts: 0.3
detr
avg_score: tensor(0.5679)
avg_ghosts: 3.3


In [87]:
total_metrics_defended = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_defended_multiplied.npy", "Defended Set", t =0.5)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 4119.05it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Defended Set...
frcnn
avg_score: tensor(0.6726)
avg_ghosts: 1.9
yolo
avg_score: tensor(0.6791)
avg_ghosts: 0.1
detr
avg_score: tensor(0.5996)
avg_ghosts: 3.8


In [76]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 0. Standard COCO Mapping (simplified for common objects)
COCO_CLASSES = {
    0: 'background', 1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 
    6: 'bus', 7: 'train', 8: 'truck', 10: 'traffic light'
} 
# Note: YOLO starts at 0 for person, Torchvision starts at 1. 
# We'll handle this offset in the visualization loop.

def plot_detections(img_pil, detections, title, ax, is_yolo=False):
    """
    detections: dict with 'boxes', 'scores', 'labels'
    is_yolo: boolean to handle YOLO's 0-indexing vs FRCNN's 1-indexing
    """
    ax.imshow(img_pil)
    ax.set_title(title)
    ax.axis('off')
    
    boxes = detections['boxes']
    scores = detections['scores']
    labels = detections['labels']

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes[i]
        score = scores[i]
        label_id = int(labels[i])
        
        # Adjusting for index mismatch if necessary
        # FRCNN/DETR: 1=person | YOLO: 0=person
        display_id = label_id if not is_yolo else label_id + 1
        label_text = COCO_CLASSES.get(display_id, f"ID:{label_id}")
        
        # Draw Box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        
        # Draw Label & Score tag
        tag = f"{label_text} {score:.2f}"
        ax.text(x1, y1 - 5, tag, color='white', fontsize=12, 
                bbox=dict(facecolor='lime', alpha=0.5, pad=1))


clean_dataset = NumpyPatchDataset("naturalistic_clean.npy")
patched_dataset = NumpyPatchDataset("naturalistic_patched.npy")
defended_dataset = NumpyPatchDataset("naturalistic_defended.npy")
updated_fusion_map = NumpyPatchDataset("naturalistic_defended_multiplied.npy")
patched_FASTERRCNN = NumpyPatchDataset("naturalistic_patched_FasterRCNN.npy")
defended_FASTERRCNN = NumpyPatchDataset("naturalistic_defended_FasterRCNN.npy")

datasets = {
    "clean": clean_dataset,
    "patched": patched_dataset,
    "defended": defended_dataset,
    "updated_fusion_map": updated_fusion_map,
    "patched_FASTERRCNN" : patched_FASTERRCNN,
    "defended_FASTERRCNN" : defended_FASTERRCNN
}

def visualize_comparison(name, idx=0):
    dataset = datasets[name]
    img_pil = dataset[idx]
    frcnn, yolo, detr, detr_proc, device = load_models()
    
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    
    # --- 1. FasterRCNN ---
    from torchvision.transforms import functional as F
    img_tensor = F.to_tensor(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        out = frcnn(img_tensor)[0]
        mask = out['scores'] > 0.7
        frcnn_res = {k: v[mask].cpu().numpy() for k, v in out.items()}
    plot_detections(img_pil, frcnn_res, "Faster R-CNN", axes[0], is_yolo=False)

    # --- 2. YOLOv8 ---
    y_res = yolo(img_pil, verbose=False, conf = 0.5)[0] 
    yolo_res = {
        'boxes': y_res.boxes.xyxy.cpu().numpy(),
        'scores': y_res.boxes.conf.cpu().numpy(),
        'labels': y_res.boxes.cls.cpu().numpy()
    }
    plot_detections(img_pil, yolo_res, "YOLOv8", axes[1], is_yolo=True)

    # --- 3. DETR ---
    inputs = detr_proc(images=img_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = detr(**inputs)
        target_sizes = torch.tensor([img_pil.size[::-1]])
        out = detr_proc.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.8)[0]
        detr_res = {k: v.cpu().numpy() for k, v in out.items()}
    plot_detections(img_pil, detr_res, "DETR", axes[2], is_yolo=False)
    plt.tight_layout()
    #plt.savefig(f"Results/comparison_{name}_{idx}.png")
    plt.show()

# dataset = NumpyPatchDataset("naturalistic_clean.npy")
# visualize_comparison(dataset, idx=0)

In [ ]:

for i in ['clean', 'patched', 'defended']:
    for j in range(10):
        visualize_comparison(i, idx=j)

In [ ]:

for i in ['updated_fusion_map']:
    for j in range(10):
        visualize_comparison(i, idx=j)

In [ ]:

for i in ['clean', 'patched_FASTERRCNN', 'defended_FASTERRCNN']:
    for j in range(10):
        visualize_comparison(i, idx=j)

In [84]:
total_metrics_patched = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_patched_FasterRCNN.npy", "Patched Set", t=0.7)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 4873.64it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Patched Set...
frcnn
avg_score: tensor(0.6489)
avg_ghosts: 3.0
yolo
avg_score: tensor(0.6821)
avg_ghosts: 0.0
detr
avg_score: tensor(0.6298)
avg_ghosts: 3.2


In [85]:
total_metrics_patched = evaluate_confidence_robustness("naturalistic_clean.npy", "naturalistic_defended_FasterRCNN.npy", "Patched Set", t=0.7)

Loading weights: 100%|██████████| 530/530 [00:00<00:00, 5694.30it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Analyzing Confidence Robustness for Patched Set...
frcnn
avg_score: tensor(0.6859)
avg_ghosts: 1.8
yolo
avg_score: tensor(0.8053)
avg_ghosts: 0.0
detr
avg_score: tensor(0.6248)
avg_ghosts: 2.6
